# 05 — Model Evaluation & Statistical Validation

Goal: Rigorous evaluation using per-class metrics, cross-validation variance analysis, and non-parametric statistical tests.

**Why statistical tests?** They validate whether observed performance differences between models are statistically significant, not due to random chance.

In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report)
from scipy.stats import wilcoxon, mannwhitneyu

X_train = sp.load_npz('../data/X_train.npz')
X_test  = sp.load_npz('../data/X_test.npz')
y_train = np.load('../data/y_train.npy')
y_test  = np.load('../data/y_test.npy')

label_names = ['Negative', 'Neutral', 'Positive']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_a = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
model_b = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
print("Models loaded.")

Models loaded.


## 5.1 Cross-Validation Summary — Logistic Regression

In [3]:
metrics = {'accuracy':[], 'f1_weighted':[], 'precision_weighted':[], 'recall_weighted':[]}

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    model_a.fit(X_tr, y_tr)
    yp = model_a.predict(X_val)
    metrics['accuracy'].append(float(accuracy_score(y_val, yp)))
    metrics['f1_weighted'].append(float(f1_score(y_val, yp, average='weighted')))
    metrics['precision_weighted'].append(float(precision_score(y_val, yp, average='weighted')))
    metrics['recall_weighted'].append(float(recall_score(y_val, yp, average='weighted')))

print(f"{'Metric':<25} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 55)
for k, v in metrics.items():
    arr = np.array(v)
    print(f"{k:<25} {arr.mean():>8.4f} {arr.std():>8.4f} {arr.min():>8.4f} {arr.max():>8.4f}")

Metric                        Mean      Std      Min      Max
-------------------------------------------------------
accuracy                    0.8265   0.0043   0.8232   0.8350
f1_weighted                 0.8275   0.0043   0.8242   0.8360
precision_weighted          0.8323   0.0041   0.8300   0.8404
recall_weighted             0.8265   0.0043   0.8232   0.8350


## 5.2 Final Holdout Test Results

In [4]:
model_a.fit(X_train, y_train)
y_pred = model_a.predict(X_test)

print("=== FINAL HOLDOUT TEST (Logistic Regression) ===")
print(f"Accuracy         : {float(accuracy_score(y_test, y_pred)):.4f}")
print(f"F1 (weighted)    : {float(f1_score(y_test, y_pred, average='weighted')):.4f}")
print(f"F1 (macro)       : {float(f1_score(y_test, y_pred, average='macro')):.4f}")
print(f"Majority baseline: {float((y_test==0).mean()):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=label_names))

=== FINAL HOLDOUT TEST (Logistic Regression) ===
Accuracy         : 0.7567
F1 (weighted)    : 0.7620
F1 (macro)       : 0.6985
Majority baseline: 0.6277

              precision    recall  f1-score   support

    Negative       0.88      0.82      0.85      1834
     Neutral       0.53      0.61      0.57       616
    Positive       0.66      0.70      0.68       472

    accuracy                           0.76      2922
   macro avg       0.69      0.71      0.70      2922
weighted avg       0.77      0.76      0.76      2922



## 5.3 Statistical Hypothesis Testing

**Wilcoxon Signed-Rank Test**: Compares paired fold scores (same folds, different models).

**Mann-Whitney U Test**: Compares score distributions between models, no normality assumption.

Null hypothesis: No significant difference between model performances.

In [5]:
scores_a, scores_b = [], []

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model_a.fit(X_tr, y_tr)
    model_b.fit(X_tr, y_tr)

    scores_a.append(float(f1_score(y_val, model_a.predict(X_val), average='weighted')))
    scores_b.append(float(f1_score(y_val, model_b.predict(X_val), average='weighted')))

scores_a = np.array(scores_a)
scores_b = np.array(scores_b)

print(f"Logistic Regression F1 per fold: {scores_a.round(4)}")
print(f"Random Forest       F1 per fold: {scores_b.round(4)}")
print()

stat_w, p_w = wilcoxon(scores_a, scores_b)
print(f"Wilcoxon Signed-Rank Test:")
print(f"  Statistic={stat_w:.4f}, p-value={p_w:.4f}")
print(f"  → {'Significant difference (p<0.05)' if p_w < 0.05 else 'No significant difference (p>=0.05)'}")
print()

stat_m, p_m = mannwhitneyu(scores_a, scores_b, alternative='two-sided')
print(f"Mann-Whitney U Test:")
print(f"  Statistic={stat_m:.4f}, p-value={p_m:.4f}")
print(f"  → {'Significant difference (p<0.05)' if p_m < 0.05 else 'No significant difference (p>=0.05)'}")

Logistic Regression F1 per fold: [0.826  0.825  0.8262 0.836  0.8242]
Random Forest       F1 per fold: [0.8787 0.8844 0.8819 0.8791 0.8742]

Wilcoxon Signed-Rank Test:
  Statistic=0.0000, p-value=0.0625
  → No significant difference (p>=0.05)

Mann-Whitney U Test:
  Statistic=0.0000, p-value=0.0079
  → Significant difference (p<0.05)


## 5.4 Per-Class Performance Across Folds

In [6]:
per_class = {n: [] for n in label_names}

for tr_idx, val_idx in skf.split(X_train, y_train):
    model_a.fit(X_train[tr_idx], y_train[tr_idx])
    yp = model_a.predict(X_train[val_idx])
    f1s = f1_score(y_train[val_idx], yp, average=None, labels=[0,1,2])
    for idx, name in enumerate(label_names):
        per_class[name].append(float(f1s[idx]))

print(f"{'Class':<12} {'Mean F1':>9} {'Std':>7}")
print("-" * 30)
for cls in label_names:
    arr = np.array(per_class[cls])
    print(f"{cls:<12} {arr.mean():>9.4f} {arr.std():>7.4f}")

print("\nObservation: Neutral class is hardest — tweets with mixed/ambiguous tone.")

Class          Mean F1     Std
------------------------------
Negative        0.8452  0.0053
Neutral         0.7990  0.0042
Positive        0.8383  0.0063

Observation: Neutral class is hardest — tweets with mixed/ambiguous tone.
